# 61. DINOv3 Patch Feature PCA와 유사도 시각화

            DINOv3의 강점은 patch-level dense feature입니다. 여기서는 동일한 shape contract를 가진 fallback feature로 PCA와 cosine similarity map을 만들어, 실제 DINOv3 feature를 분석할 때 어떤 방식으로 볼지 익힙니다.


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path("Deeplearning") / "Vision 기초" / "8장",
    Path("Vision 기초") / "8장",
]
NOTEBOOK_DIR = next((p for p in candidates if (p / "seg8_utils.py").exists()), Path.cwd())
sys.path.append(str(NOTEBOOK_DIR))

from seg8_utils import *
set_korean_font()
set_seed(7)

DATA_ROOT = ensure_dataset()
RUNS_ROOT = NOTEBOOK_DIR / "runs"


## 61-1. Feature map 추출


In [ ]:
from PIL import Image
import torch

pairs = list_pairs(DATA_ROOT, "val")
image = Image.open(pairs[1][0]).convert("RGB")
target = np.asarray(Image.open(pairs[1][1]), dtype=np.int64)
image_np = np.asarray(image)
image_tensor = image_to_tensor(image_np).unsqueeze(0)

backbone = DenseFeatureAdapter("handcrafted")
with torch.no_grad():
    fmap = backbone(image_tensor)[0].permute(1, 2, 0).numpy()

fmap.shape


## 61-2. PCA RGB 시각화


In [ ]:
h, w, c = fmap.shape
x = fmap.reshape(-1, c)
x = x - x.mean(axis=0, keepdims=True)
_, _, vt = np.linalg.svd(x, full_matrices=False)
pca = x @ vt[:3].T
pca = (pca - pca.min(axis=0)) / (pca.max(axis=0) - pca.min(axis=0) + 1e-6)
pca_img = pca.reshape(h, w, 3)

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(image_np)
axes[0].set_title("image")
axes[1].imshow(colorize_mask(target))
axes[1].set_title("target")
axes[2].imshow(pca_img)
axes[2].set_title("feature PCA")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


## 61-3. 한 patch와 전체 patch의 cosine similarity


In [ ]:
cy, cx = h // 2, w // 2
feat = fmap.reshape(-1, c)
query = fmap[cy, cx]
sim = feat @ query / (np.linalg.norm(feat, axis=1) * np.linalg.norm(query) + 1e-6)
sim = sim.reshape(h, w)

plt.figure(figsize=(4, 4))
plt.imshow(image_np)
plt.imshow(sim, cmap="magma", alpha=0.55)
plt.scatter([cx], [cy], c="cyan", s=40)
plt.title("cosine similarity map")
plt.axis("off")
plt.show()
